In [2]:
!pip install h5py

   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.9 MB ? eta -:--:--
   --------------------- ------------------ 1.6/2.9 MB 4.9 MB/s eta 0:00:01
   -------------------------------- ------- 2.4/2.9 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 4.2 MB/s eta 0:00:00


In [3]:
#!/usr/bin/env python3
import os
import numpy as np
import pandas as pd
import h5py

# ─── CONFIG ────────────────────────────────────────────────────────────────
CSV_PATH   = 'please.csv'     # your input CSV
OUTPUT_DIR = 'hdf5MF'        # where to write plane_*.h5 files

# ─── HELPERS ────────────────────────────────────────────────────────────────
def ordinal(n: int) -> str:
    """Return an ordinal string, e.g. 1→'1st', 2→'2nd', 3→'3rd', 4→'4th', etc."""
    if 10 <= (n % 100) <= 20:
        suffix = 'th'
    else:
        suffix = {1:'st',2:'nd',3:'rd'}.get(n % 10, 'th')
    return f"{n}{suffix}"

def reshape_to_15x15(vals: np.ndarray) -> np.ndarray:
    """
    Take a 1×193 sensor vector and return a 15×15 array
    with the same concentric‐rings + 7×15 middle logic.
    """
    ring_counts = [7,11,13,13]
    offsets     = [4, 2,  1,  1]
    # mirror for inner rings
    counts_full = ring_counts + ring_counts[::-1]
    offs_full   = offsets     + offsets[::-1]

    assert vals.size == sum(counts_full) + 7*15, \
        f"Expected 193 sensors, got {vals.size}"

    # build the 8 ring‐rows
    ring_rows = [np.zeros(15) for _ in range(8)]
    idx = 0
    for i,(cnt,off) in enumerate(zip(counts_full, offs_full)):
        ring_rows[i][off:off+cnt] = vals[idx:idx+cnt]
        idx += cnt

    # center block (7×15) from the remaining values
    mid_block = vals[idx:]
    mid_mat   = mid_block.reshape((7,15))

    # stack: top 4, mid 7, bottom 4
    return np.vstack(ring_rows[:4] + [mid_mat] + ring_rows[4:])

# ─── MAIN PIPELINE ────────────────────────────────────────────────────────
def main():
    # 1) load CSV
    df       = pd.read_csv(CSV_PATH)
    times    = df.iloc[:,0].to_numpy()    # shape (T,)
    sensors  = df.iloc[:,1:].to_numpy()   # shape (T, N)

    # 2) compute number of 193‐sensor planes
    T, N = sensors.shape
    if N % 193 != 0:
        raise ValueError(f"{N=} is not a multiple of 193")
    P = N // 193

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 3) loop over planes
    for p in range(P):
        plane_slice = sensors[:, p*193 : (p+1)*193]  # (T,193)

        # stack into (T,15,15)
        arr3d = np.zeros((T,15,15))
        for t in range(T):
            arr3d[t] = reshape_to_15x15(plane_slice[t])

        # decide a human‐readable label
        if p == 0:
            label = "highest"
        elif p == P-1:
            label = "lowest"
        else:
            label = f"{ordinal(p+1)}_highest"

        fname = f"plane_{p+1:02d}_{label}.h5"
        fpath = os.path.join(OUTPUT_DIR, fname)

        # write HDF5
        with h5py.File(fpath, 'w') as h5:
            h5.create_dataset('time', data=times)
            h5.create_dataset('data', data=arr3d)
        print(f"Saved {fpath} (time: {times.shape}, data: {arr3d.shape})")

    # 4) report on HDF5 contents
    print("\n=== HDF5 STRUCTURE ===")
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        if not fname.endswith('.h5'):
            continue
        fpath = os.path.join(OUTPUT_DIR, fname)
        print(f"\n-- {fpath} --")
        with h5py.File(fpath,'r') as h5:
            for dname, dset in h5.items():
                print(f"  /{dname}  shape={dset.shape}  dtype={dset.dtype}")

main()

Saved hdf5MF\plane_01_highest.h5 (time: (5000,), data: (5000, 15, 15))
Saved hdf5MF\plane_02_2nd_highest.h5 (time: (5000,), data: (5000, 15, 15))
Saved hdf5MF\plane_03_3rd_highest.h5 (time: (5000,), data: (5000, 15, 15))
Saved hdf5MF\plane_04_4th_highest.h5 (time: (5000,), data: (5000, 15, 15))
Saved hdf5MF\plane_05_5th_highest.h5 (time: (5000,), data: (5000, 15, 15))
Saved hdf5MF\plane_06_6th_highest.h5 (time: (5000,), data: (5000, 15, 15))
Saved hdf5MF\plane_07_7th_highest.h5 (time: (5000,), data: (5000, 15, 15))
Saved hdf5MF\plane_08_8th_highest.h5 (time: (5000,), data: (5000, 15, 15))
Saved hdf5MF\plane_09_lowest.h5 (time: (5000,), data: (5000, 15, 15))

=== HDF5 STRUCTURE ===

-- hdf5MF\plane_01_highest.h5 --
  /data  shape=(5000, 15, 15)  dtype=float64
  /time  shape=(5000,)  dtype=float64

-- hdf5MF\plane_02_2nd_highest.h5 --
  /data  shape=(5000, 15, 15)  dtype=float64
  /time  shape=(5000,)  dtype=float64

-- hdf5MF\plane_03_3rd_highest.h5 --
  /data  shape=(5000, 15, 15)  dty